# WGAN-GP for synthetic surface-defect images

Defective samples are rare on a production line, so this notebook trains a **Wasserstein GAN with gradient penalty**
to synthesise additional 128×128 grayscale line-scan crops, then keeps only the samples that an independent
detector considers defective.

* **Generator**: DCGAN-style transposed-convolution stack, z ∈ ℝ¹⁰⁰ → 1×128×128 (tanh).
* **Critic**: strided convolutions + LeakyReLU, no sigmoid (Wasserstein critic).
* **Loss**: `E[D(G(z))] − E[D(x)] + λ·GP` with λ = 10; generator minimises `−E[D(G(z))]`; Adam (β₁ = 0.5),
  critic learning rate 0.1× the generator's.
* **Filtering** (last section): every generated image is scored with the learning-free reference-image detector from
  the *one-shot defect detection* project (MSE + SSIM against a "defect-erased" version of the same image) and a
  One-Class SVM trained on normal images; only images flagged as anomalous are saved as synthetic defects.

**Results.** No quantitative generation metric (e.g. FID) was computed; the outputs only contain the training log.
Generated and real samples were inspected visually (images removed here because they show proprietary products).
Known limitation: the critic still uses BatchNorm, which the WGAN-GP paper advises against (LayerNorm/no norm is preferred).

**Data:** proprietary line-scan crops (not included).


In [ ]:
#%matplotlib inline
import os
import random
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import os

In [ ]:
import os

# Root folder of the dataset(s) used in this notebook.
# Set the DATA_ROOT environment variable or edit the default below.
# On Google Colab you can first mount Drive with:
#   from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

import shutil

# Training images: DATA_ROOT/gan_training/<class>/*.png (ImageFolder layout)
img_dir_good = f"{DATA_ROOT}/gan_training/good"
img_dir_bad = f"{DATA_ROOT}/gan_training/bad"
local_train_dir = "work/gan_train"
for d in (img_dir_good, img_dir_bad):
    shutil.copytree(d, f"{local_train_dir}/{d.rstrip('/').split('/')[-1]}", dirs_exist_ok=True)

In [ ]:
# %%
# Root directory for dataset - CHANGE THIS TO YOUR IMAGE FOLDER PATH
# Assuming your images are in a folder structure like dataroot/class_name/image.jpg
dataroot = local_train_dir

# Number of workers for dataloader
workers = 2

# Batch size during training
batch_size = 32

# Spatial size of training images. All images will be resized to this
#   size using a transformer.
image_size = 128 # Keep this consistent with your updated Generator/Discriminator

# Number of channels in the training images. For color images this is 3
# Set to 3 for color images, 1 for grayscale
nc = 1 # Assuming your images are grayscale based on previous code

# Size of z latent vector (i.e. size of generator input)
nz = 100

# Size of feature maps in generator
ngf = 64

# Size of feature maps in discriminator
ndf = 64

# Number of training epochs
num_epochs = 200

# Learning rate for optimizers
lr = 0.0002

# Beta1 hyperparameter for Adam optimizers
beta1 = 0.5

# Number of GPUs available. Use 0 for CPU mode.
ngpu = 1

# %%
# Data loading and augmentation

# Define the transformations to apply to the images
# Including resizing, converting to tensor, and normalization
# Add augmentation transforms here for GAN training stability on small data
transform = transforms.Compose([
    transforms.Resize(image_size), # Resize images to the target size
    transforms.CenterCrop(image_size), # Optional: Crop to ensure square size if images aren't already square
    transforms.Grayscale(num_output_channels=nc) if nc == 1 else transforms.Lambda(lambda x: x), # Convert to grayscale if nc=1
    transforms.RandomHorizontalFlip(), # Example augmentation

    # Add more augmentations here if needed
    transforms.ToTensor(), # Convert PIL Image to PyTorch Tensor
    transforms.Normalize((0.5,), (0.5,)) if nc == 1 else transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), # Normalize to [-1, 1]
])

# Create the dataset using ImageFolder
# ImageFolder expects the root folder to contain subfolders for each class
# e.g., dataroot/class_0/img1.png, dataroot/class_1/img2.png
# If all your training images are in a single folder under dataroot,
# ImageFolder will treat that folder as a single class.
dataset = dset.ImageFolder(root=dataroot,
                           transform=transform)

In [ ]:
# Create the dataloader
# Set shuffle=True to shuffle the data each epoch
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                         shuffle=True, num_workers=workers)

# Decide which device we want to run on
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

# Plot some training images
# The data from ImageFolder will be a tuple (image, label)
real_batch = next(iter(dataloader))
plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Training Images")
# The tensor is real_batch[0], take the first 64 images
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(),(1,2,0)))
plt.show()

In [ ]:
# custom weights initialization called on ``netG`` and ``netD``
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
# Generator network, designed to output 128x128 images

class Generator(nn.Module):
    def __init__(self, ngpu):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            # 1x1 -> 4x4
            nn.ConvTranspose2d( nz, ngf * 8, 4, 1, 0, bias=False), # Adjusted ngf multiplier
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 4 x 4

            # 4x4 -> 8x8
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 8 x 8

            # 8x8 -> 16x16
            nn.ConvTranspose2d( ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 16 x 16

            # 16x16 -> 32x32
            nn.ConvTranspose2d( ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. (ngf) x 32 x 32

            # 32x32 -> 64x64
            nn.ConvTranspose2d( ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),
            # state size. (ngf/2) x 64 x 64

            # 64x64 -> 128x128
            nn.ConvTranspose2d( ngf // 2, nc, 4, 2, 1, bias=False), # Final layer for 128x128 output
            nn.Tanh()
            # state size. (nc) x 128 x 128
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
# Create the generator
netG = Generator(ngpu).to(device)

# Handle multi-GPU if desired
if (device.type == 'cuda') and (ngpu > 1):
    netG = nn.DataParallel(netG, list(range(ngpu)))

# Apply the ``weights_init`` function to randomly initialize all weights
#  to ``mean=0``, ``stdev=0.02``.
netG.apply(weights_init)

# Print the model
print(netG)

Generator(
  (main): Sequential(
    (0): ConvTranspose2d(100, 512, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (13): BatchNorm2d(

In [ ]:
# Discriminator network, designed to accept 128x128 images

class Discriminator(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is (nc) x 128 x 128
            # 128x128 -> 64x64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf) x 64 x 64

            # 64x64 -> 32x32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*2) x 32 x 32

            # 32x32 -> 16x16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*4) x 16 x 16

            # 16x16 -> 8x8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*8) x 8 x 8

            # 8x8 -> 4x4
            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False), # Adjusted ndf multiplier
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*16) x 4 x 4

            # 4x4 -> 1x1
            nn.Conv2d(ndf * 16, 1, 4, 1, 0, bias=False), # Input is now 4x4, kernel is 4x4, stride 1, padding 0 -> 1x1 output
            # state size: 1 x 1 x 1


        )

    def forward(self, input):
        return self.main(input)

In [ ]:
# Create the Discriminator
netD = Discriminator(ngpu).to(device)

# Handle multi-GPU if desired
if (device.type == 'cuda') and (ngpu > 1):
    netD = nn.DataParallel(netD, list(range(ngpu)))

# Apply the ``weights_init`` function to randomly initialize all weights
# like this: ``to mean=0, stdev=0.2``.
netD.apply(weights_init)

# Print the model
print(netD)

Discriminator(
  (main): Sequential(
    (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (9): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Conv2d(512, 1024, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (12): BatchNorm2d(1024, eps=1e-05

In [ ]:
# Add this function definition before the training loop
def calculate_gradient_penalty(critic, real_images, fake_images, device, lambda_gp):
        """Calculates the gradient penalty for WGAN-GP."""
        # Get random interpolation between real and fake samples
        alpha = torch.rand(real_images.size(0), 1, 1, 1, device=device)
        interpolates = (alpha * real_images + ((1 - alpha) * fake_images)).requires_grad_(True)

        # Get critic scores for interpolated samples
        critic_interpolates = critic(interpolates)

        # Compute the gradient of critic_interpolates with respect to interpolates
        gradients = torch.autograd.grad(outputs=critic_interpolates, inputs=interpolates,
                                      grad_outputs=torch.ones_like(critic_interpolates),
                                      create_graph=True, retain_graph=True, only_inputs=True)[0]

        # Flatten the gradients and compute the L2 norm
        gradients = gradients.view(gradients.size(0), -1)
        gradient_norm = gradients.norm(2, dim=1)

        # Compute the gradient penalty
        gradient_penalty = ((gradient_norm - 1) ** 2).mean() * lambda_gp
        return gradient_penalty

In [ ]:
# Create batch of latent vectors that we will use to visualize
#  the progression of the generator
fixed_noise = torch.randn(64, nz, 1, 1, device=device)



# Setup Adam optimizers for both G and D
# Reduce the learning rate for the Discriminator
lr_D = lr * 0.1  # slower critic for stability on a small dataset
optimizerD = optim.Adam(netD.parameters(), lr=lr_D, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

In [ ]:
# Lists to keep track of progress
img_list = []
G_losses = []
D_losses = []
iters = 0

# Define how many times to train the generator per discriminator update
generator_train_ratio = 1  # Start with 1:1 ratio for stability
# Define the coefficient for the gradient penalty
lambda_gp = 10 # Common value for gradient penalty

print("Starting Training Loop...")
# For each epoch
for epoch in range(num_epochs):
    # For each batch in the dataloader
    for i, data in enumerate(dataloader, 0):

        ############################
        # (1) Update D network: maximize D(x) - D(G(z)) + Gradient Penalty
        ###########################
        netD.zero_grad()

        # Format batch
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)

        # Forward pass real batch through D (Critic)
        output_real = netD(real_cpu).view(-1)

        # Generate fake image batch with G
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)

        # Forward pass fake batch through D (Critic)
        output_fake = netD(fake.detach()).view(-1) # Detach fake here

        # Calculate Wasserstein loss for D
        errD_wasserstein = output_fake.mean() - output_real.mean()

        # Calculate gradient penalty
        gp = calculate_gradient_penalty(netD, real_cpu.data, fake.data, device, lambda_gp)

        # Compute total D loss
        errD = errD_wasserstein + gp

        # Calculate gradients for D and update
        errD.backward()
        optimizerD.step()

        # Record D(x) and D(G(z)) values for logging (optional, but helpful)
        D_x = output_real.mean().item()
        D_G_z1 = output_fake.mean().item()


        ############################
        # (2) Update G network: minimize -D(G(z))
        ###########################

        # Train generator multiple times
        if i % generator_train_ratio == 0: # Train generator every generator_train_ratio D updates
            netG.zero_grad()
            # Generate a NEW batch of fake images for each G update
            noise = torch.randn(b_size, nz, 1, 1, device=device)
            fake = netG(noise)

            # Pass the NEW fake batch through D
            output_fake_for_G = netD(fake).view(-1)

            # Calculate G's loss based on this output (minimize -D(G(z)))
            errG = -output_fake_for_G.mean() # Wasserstein loss for G

            # Calculate gradients for G and update
            errG.backward()
            optimizerG.step()

            # Record D(G(z)) for logging after G update
            D_G_z2 = output_fake_for_G.mean().item()
        else:
            # If not training G in this iteration, just copy the previous D_G_z2 for logging consistency
            D_G_z2 = D_G_z1 # Or set to None or a placeholder if preferred

        # Output training stats
        if i % 50 == 0:
              # Adjust the print statement to reflect WGAN-GP losses and values
              print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f\tGP: %.4f'
                    % (epoch, num_epochs, i, len(dataloader),
                      errD.item(), errG.item() if 'errG' in locals() else float('nan'),
                      D_x, D_G_z1, D_G_z2 if 'D_G_z2' in locals() else float('nan'),
                      gp.item()))


        # Save Losses for plotting later
        # Note: These losses are recorded only after the last G update for this batch
        if 'errG' in locals(): # Only append G_loss if G was trained in this iteration
            G_losses.append(errG.item())
            D_losses.append(errD.item())
        else: # If G was not trained, append NaN or the last recorded losses
              G_losses.append(G_losses[-1] if G_losses else float('nan'))
              D_losses.append(errD.item()) # Always append D loss


        # Check how the generator is doing by saving G's output on fixed_noise
        if (iters % 500 == 0) or ((epoch == num_epochs-1) and (i == len(dataloader)-1)):
            with torch.no_grad():
                fake = netG(fixed_noise).detach().cpu()
            img_list.append(vutils.make_grid(fake, padding=2, normalize=True))

        iters += 1

Starting Training Loop...
[0/200][0/32]	Loss_D: 1794.7832	Loss_G: -0.7061	D(x): 0.5432	D(G(z)): 0.3980 / 0.7061	GP: 1794.9285
[1/200][0/32]	Loss_D: 123.3213	Loss_G: -71.8413	D(x): 7.8345	D(G(z)): 72.5273 / 71.8413	GP: 58.6284
[2/200][0/32]	Loss_D: 69.0733	Loss_G: -65.2978	D(x): 16.9419	D(G(z)): 67.0751 / 65.2978	GP: 18.9401
[3/200][0/32]	Loss_D: 39.4579	Loss_G: -52.9374	D(x): 25.2813	D(G(z)): 55.1900 / 52.9374	GP: 9.5492
[4/200][0/32]	Loss_D: 20.9964	Loss_G: -44.8282	D(x): 31.9243	D(G(z)): 46.8539 / 44.8282	GP: 6.0669
[5/200][0/32]	Loss_D: 8.5297	Loss_G: -43.0166	D(x): 40.4218	D(G(z)): 44.8110 / 43.0166	GP: 4.1405
[6/200][0/32]	Loss_D: 5.8651	Loss_G: -40.3396	D(x): 44.3446	D(G(z)): 42.8358 / 40.3396	GP: 7.3739
[7/200][0/32]	Loss_D: 5.2117	Loss_G: -43.5135	D(x): 48.3705	D(G(z)): 45.6289 / 43.5135	GP: 7.9534
[8/200][0/32]	Loss_D: 4.0343	Loss_G: -48.0405	D(x): 50.4920	D(G(z)): 49.9114 / 48.0405	GP: 4.6149
[9/200][0/32]	Loss_D: 1.7171	Loss_G: -49.6606	D(x): 52.9904	D(G(z)): 51.1925 / 49.66

In [ ]:
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses,label="G")
plt.plot(D_losses,label="D")
plt.xlabel("iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# Grab a batch of real images from the dataloader
real_batch = next(iter(dataloader))

# Plot the real images
plt.figure(figsize=(15,15))
plt.subplot(1,2,1)
plt.axis("off")
plt.title("Real Images")
plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=5, normalize=True).cpu(),(1,2,0)))

# Plot the fake images from the last epoch
plt.subplot(1,2,2)
plt.axis("off")
plt.title("Fake Images")
plt.imshow(np.transpose(img_list[-1],(1,2,0)))
plt.show()

In [ ]:
# Save the trained generator and critic
os.makedirs("outputs", exist_ok=True)
torch.save(netG.state_dict(), "outputs/wgan_gp_generator.pth")
torch.save(netD.state_dict(), "outputs/wgan_gp_critic.pth")

In [ ]:
# -*- coding: utf-8 -*-
"""
Benchmarking script for One-Class SVM inference.
Assumes training is already completed and the model/scaler are available.
"""

import time
import numpy as np
import cv2
import os
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
import pickle # Or joblib, for saving/loading model and scaler
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
# --- Assume compute_image_distance is defined ---
# You should include your compute_image_distance function here or ensure it's imported
# if it's in a separate file. For this script, we'll include a placeholder.
def compute_image_distance(image1, image2, metric='mse'):
    """Computes the distance between two images based on the chosen metric.

    Args:
        image1: The first image as a NumPy array.
        image2: The second image as a NumPy array.
        metric: The distance metric to use ('psnr', 'ssim', 'mse'). Defaults to 'mse'.

    Returns:
        The computed distance between the two images. Returns -1 if an error occurs
        or if the metric is not supported.
    """
    if image1.shape != image2.shape:
        print("Error: Images must have the same dimensions.")
        return -1

    if metric.lower() == 'mse':
        mse = np.sum((image1.astype(np.int32) - image2.astype(np.int32)) ** 2)
        return mse
    elif metric.lower() == 'psnr':
        psnr_score = psnr(image1, image2, data_range=255)
        return psnr_score
    elif metric.lower() == 'ssim':
        # Use OpenCV's built-in SSIM function
        ssim_score = ssim(image1, image2)
        return ssim_score
    else:
        print(f"Error: Unsupported metric '{metric}'.")
        return -1


# --- Assume find_white_stripe_borders is defined ---
# Include your find_white_stripe_borders function here or ensure it's imported.

def find_white_stripe_borders(image):
    """Finds the borders of a white stripe in an image.

    Args:
        image: The input image as a NumPy array.
        tolerance: The intensity difference allowed for a pixel to be
                   considered part of the stripe.

    Returns:
        A tuple containing the upper and lower border indices of the stripe.
    """
    row_means = np.mean(image, axis=1)
    center_row = np.argmax(row_means[40:60])+ 40
    threshold = row_means[center_row]-40 # Intensity of the center row (stripe)

    # Find upper border
    upper_border = center_row-5
    while upper_border < center_row  and row_means[upper_border] <  threshold :
        upper_border += 1

    # Find lower border
    lower_border = center_row+5
    while lower_border > center_row and row_means[lower_border]  < threshold :
        lower_border -= 1

    return (upper_border, lower_border,center_row)

# --- Assume the image processing steps (3rd quartile, dilation, erosion) are defined ---
# Include your image processing functions here or ensure they are imported.
def process_image(image):
    """Applies the 3rd quartile row processing, dilation, and erosion."""
    # Ensure image is grayscale if needed
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    temp_img = image.copy()
    for i in range(temp_img.shape[0]):
        # Handle potential empty row for quantile calculation if image width is 0
        if temp_img.shape[1] > 0:
             quartile_val = np.quantile(temp_img[i], 0.75)
             temp_img[i] = quartile_val
        else:
             temp_img[i] = 0 # Set to 0 if row is empty

    kernel_size = (7, 7)
    kernel = np.ones(kernel_size, np.uint8)

    # Ensure image is of type uint8 before dilation/erosion
    if temp_img.dtype != np.uint8:
        temp_img = temp_img.astype(np.uint8)

    temp_img = cv2.dilate(temp_img, kernel, iterations=5)
    temp_img = cv2.erode(temp_img, kernel, iterations=5)




    return temp_img

def apply_border_replacement(original_image, modified_image):
    """Replaces areas outside borders with the original image and sets the border area to white."""
    # Ensure images are grayscale if needed
    if len(original_image.shape) == 3:
        original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2GRAY)
    if len(modified_image.shape) == 3:
        modified_image = cv2.cvtColor(modified_image, cv2.COLOR_BGR2GRAY)

    modified_image_copy = modified_image.copy() # Work on a copy

    upper_border, lower_border, center_row = find_white_stripe_borders(modified_image_copy)

    if upper_border != -1 and lower_border != -1:
        # Ensure slicing is valid
        upper_slice = slice(0, min(lower_border, original_image.shape[0]))
        lower_slice = slice(max(0, upper_border), original_image.shape[0])
        border_slice = slice(max(0, upper_border), min(lower_border, original_image.shape[0]))


        modified_image_copy[upper_slice, :] = original_image[upper_slice, :]
        modified_image_copy[lower_slice, :] = original_image[lower_slice, :]

        # Ensure border_slice is valid before creating white_border
        if border_slice.start < border_slice.stop:
            border_shape = modified_image_copy[border_slice, :].shape
            # Ensure the shape is valid (height, width) and width is > 0
            if border_shape[0] > 0 and border_shape[1] > 0:
                white_border = np.ones(border_shape) * np.max(original_image)
                modified_image_copy[border_slice, :] = white_border

    return modified_image_copy


# --- Configuration ---
# Replace with the actual path to your test dataset
test_data_root_dir = f'{DATA_ROOT}/line_scan_crops/test'
train_data_root_dir= f'{DATA_ROOT}/line_scan_crops/train'
# --- Function to load images (assuming you have a function for this) ---
def load_images_with_labels(root_dir):
    """Loads images and labels from a directory structure (e.g., good/bad subfolders)."""
    images = []
    labels = []
    for label, subdir in enumerate(sorted(os.listdir(root_dir))):
        subdir_path = os.path.join(root_dir, subdir)
        if os.path.isdir(subdir_path):
            for filename in os.listdir(subdir_path):
                if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                    img_path = os.path.join(subdir_path, filename)
                    # Load as grayscale for consistency with your processing
                    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        images.append(img)
                        labels.append(label) # 0 for first subdir (e.g., 'good'), 1 for second (e.g., 'bad')
                    else:
                        print(f"Error loading image: {img_path}")
    return images, labels

# --- Load Test Data ---
print(f"Loading test images from: {test_data_root_dir}")
test_original_images, test_true_labels = load_images_with_labels(test_data_root_dir)
train_original_images, train_true_labels = load_images_with_labels(train_data_root_dir)
print(f"Loaded {len(test_original_images)} test images.")
# Assuming 'images' contains the original images from the test set.
third_quartile_images2 = []

for img_array2 in train_original_images:
  temp_img2 = img_array2.copy()  # Create a copy of the current image
  for i in range(temp_img2.shape[0]): # Iterate through each row
    # Calculate the 3rd quartile value for the current row
    quartile_val2 = np.quantile(temp_img2[i], 0.75)

    # Set all pixels in the row to the 3rd quartile value
    temp_img2[i] = quartile_val2

  # Create kernel
  kernel_size = (7, 7)
  kernel = np.ones(kernel_size, np.uint8)

  # Apply dilation to the temporary image
  temp_img2 = cv2.dilate(temp_img2.astype(np.uint8), kernel, iterations=5)
  temp_img2 = cv2.erode(temp_img2.astype(np.uint8), kernel, iterations=5)

  # Append the modified temporary image to modified_img list
  third_quartile_images2.append(temp_img2)


for i in range(len(train_original_images)):
    original_image2 = train_original_images[i]
    # Get a copy of the modified image to avoid changing the original list
    modified_image2 = third_quartile_images2[i].copy()

    upper_border2, lower_border2,center_row2 = find_white_stripe_borders(modified_image2)

    # Check if borders were found
    if upper_border2 != -1 and lower_border2 != -1:
        # Replace areas outside borders with original image
        modified_image2[:lower_border2, :] = original_image2[:lower_border2, :]
        modified_image2[upper_border2:, :] = original_image2[upper_border2:, :]
        # Get the shape of the border region slice
        border_shape2 = modified_image2[upper_border2:lower_border2, :].shape

        #  Create an array of ones with the same shape and multiply by 255
        white_border2 = np.ones(border_shape2) * np.max(original_image2)

        # Assign the white border to the modified image, correcting the column slice
        modified_image2[upper_border2:lower_border2, :] = white_border2
    # Update the modified image in the list
    third_quartile_images2[i] = modified_image2
if not test_original_images:
    print("No test images loaded. Exiting benchmarking.")
    exit()

def prepare_distance_dataset(original_images, modified_images):
    """
    Computes MSE and SSIM distances between original and modified images
    and prepares a dataset for the One-Class SVM.

    Args:
        original_images: List of original images (NumPy arrays).
        modified_images: List of modified images (NumPy arrays).

    Returns:
        A NumPy array where each row contains [MSE distance, SSIM distance]
        for a pair of images.
    """
    distances = []
    for i in range(len(original_images)):
        # Compute MSE and SSIM distances
        distance_mse = compute_image_distance(original_images[i], modified_images[i], metric='mse')
        distance_ssim = compute_image_distance(original_images[i], modified_images[i], metric='ssim')

        # Handle potential infinite or NaN values that can occur with PSNR/SSIM
        # For One-Class SVM, it's better to have finite values.
        # Replace infinite MSE with a large value
        if np.isinf(distance_mse):
            distance_mse = 1e9  # Use a large finite number
        # Replace NaN or problematic SSIM values (if any)
        if np.isnan(distance_ssim) or distance_ssim is None:
             distance_ssim = 0.0 # Or another appropriate default

        distances.append([distance_mse, distance_ssim])

    return np.array(distances)

def train_and_evaluate_one_class_svm(train_distance_dataset,nu,gamma):
    """
    Trains a One-Class SVM on the training distance dataset and evaluates it
    on the testing distance dataset.

    Args:
        train_distance_dataset: NumPy array of distances for training.
                                Should contain data from the normal class only.
        test_distance_dataset: NumPy array of distances for testing.
                                Can contain data from both normal and anomaly classes.
        test_labels: True labels for the testing dataset.

    Returns:
        A tuple containing:
            one_class_clf: The trained One-Class SVM classifier.
            scaler: The StandardScaler object used for scaling.
    """
    # --- Scale Training and Testing Data ---
    scaler = StandardScaler()
    train_distance_scaled = scaler.fit_transform(train_distance_dataset)

    # --- Train One-Class SVM ---
    one_class_clf = OneClassSVM(kernel='rbf', nu=nu, gamma=gamma) # Adjust nu as needed
    one_class_clf.fit(train_distance_scaled)


    return one_class_clf, scaler

train_distances = prepare_distance_dataset(train_original_images, third_quartile_images2) # Assuming images2/third_quartile_images2 are your training data (normal class)

# 2. Train and evaluate the One-Class SVM
one_class_svm_clf, scaler = train_and_evaluate_one_class_svm(train_distances,nu=.01,gamma=1)
# -----------------------------------------------------------------------------

# --- Benchmarking Inference Time ---
print("\nStarting inference benchmarking...")
start_time_inference = time.time()

test_modified_images = []
test_distances = []

for i, original_img in enumerate(test_original_images):
    # 1. Apply image processing steps
    modified_img = process_image(original_img)
    modified_img_with_borders = apply_border_replacement(original_img, modified_img)
    test_modified_images.append(modified_img_with_borders)

    # 2. Compute distance features
    distance_mse = compute_image_distance(original_img, modified_img_with_borders, metric='mse')
    distance_ssim = compute_image_distance(original_img, modified_img_with_borders, metric='ssim')

    # Handle potential infinite or NaN values
    if np.isinf(distance_mse):
        distance_mse = 1e9
    if np.isnan(distance_ssim) or distance_ssim is None:
         distance_ssim = 0.0

    test_distances.append([distance_mse, distance_ssim])

# Convert distances to NumPy array
test_distances_array = np.array(test_distances)

# 3. Scale distances using the pre-fitted scaler
test_distance_scaled = scaler.transform(test_distances_array)

# 4. Make predictions using the trained One-Class SVM
test_predictions_raw = one_class_svm_clf.predict(test_distance_scaled)
predicted_labels = np.where(test_predictions_raw == 1, 1, 0) # Map to your labels

end_time_inference = time.time()
inference_time_taken = end_time_inference - start_time_inference

# --- Print Results ---
num_test_images = len(test_original_images)
# --- Generation, Classification, and Saving Loop ---
print("\nStarting image generation and classification...")
gan_output_dir=f'{DATA_ROOT}/gan_generated'
bad_images_count = 0
max_bad_images = 500
generated_bad_images_dir = os.path.join(gan_output_dir, "generated_bad_images")
os.makedirs(generated_bad_images_dir, exist_ok=True)

while bad_images_count < max_bad_images:
    # Generate images using the GAN Generator (netG)
    # Generate a batch of latent vectors as input to the generator
    num_images_to_generate_batch = 16 # You can adjust this batch size
    noise = torch.randn(num_images_to_generate_batch, nz, 1, 1, device=device)

    # Generate fake images from the noise
    with torch.no_grad(): # Turn off gradient tracking for inference
        fake_images_tensor = netG(noise).detach().cpu()

    print(f"Generated a batch of {num_images_to_generate_batch} images with GAN. Classifying...")

    # Process and classify each generated image in the batch
    for j in range(fake_images_tensor.size(0)):
        if bad_images_count >= max_bad_images:
            break # Stop if we have enough bad images

        # Get a single generated image tensor
        generated_img_tensor = fake_images_tensor[j]

        # Convert the PyTorch tensor to a PIL Image
        # Normalize the image back to [0, 1] for PIL
        generated_img_pil = transforms.ToPILImage()(vutils.make_grid(generated_img_tensor, normalize=True))


        # Convert PIL image to OpenCV format (NumPy array)
        generated_img_cv2 = cv2.cvtColor(np.array(generated_img_pil), cv2.COLOR_RGB2GRAY)
        # 1. Apply image processing steps
        modified_img = process_image(generated_img_cv2)
        modified_img_with_borders = apply_border_replacement(generated_img_cv2, modified_img)

        # 2. Compute distance features
        distance_mse = compute_image_distance(generated_img_cv2, modified_img_with_borders, metric='mse')
        distance_ssim = compute_image_distance(generated_img_cv2, modified_img_with_borders, metric='ssim')

        # Handle potential infinite or NaN values
        if np.isinf(distance_mse):
            distance_mse = 1e9
        if np.isnan(distance_ssim) or distance_ssim is None:
             distance_ssim = 0.0

        # Create a feature array for this image
        image_features = np.array([[distance_mse, distance_ssim]])

        # 3. Scale distances using the pre-fitted scaler
        image_features_scaled = scaler.transform(image_features)

        # 4. Make prediction using the trained One-Class SVM
        prediction_raw = one_class_svm_clf.predict(image_features_scaled)
        predicted_label = np.where(prediction_raw == 1, 1, 0) # 1 for normal, 0 for anomaly

        # Check if the image is classified as 'bad' (anomaly)
        if predicted_label[0] == 0:
            # Save the 'bad' image
            bad_images_count += 1
            save_path = os.path.join(generated_bad_images_dir, f"bad_image_{bad_images_count:04d}.png")
            generated_img_pil.save(save_path)
            print(f"Saved bad image {bad_images_count}/{max_bad_images} to {save_path}")
        # else:
            # Optional: Print if an image is classified as good
            # print("Generated image classified as good.")

print(f"\nFinished generating and saving {bad_images_count} bad images.")

Loading test images from: <DATA_ROOT>/line_scan_crops/test
Loaded 1000 test images.
Starting inference benchmarking...
Starting image generation and classification...
Finished generating and saving 500 bad images.
